# S07 Prévision et évaluation simples de séries chronologiques avec darts

### Introduction

La prévision de la demande par séries chronologiques est essentielle à une gestion efficace de la chaîne d'approvisionnement. Grâce à des prévisions de la demande précises, les entreprises peuvent optimiser leurs stocks, améliorer la planification de la demande, affiner les prévisions de ventes et atténuer les risques de la chaîne d'approvisionnement.

Avec les développements récents en Python et dans le logiciel libre, il existe de nombreux paquets simples d'utilisation comme [statsmodels](https://www.statsmodels.org/stable/index.html) (surtout pour les techniques statistiques), [Prophet (de Facebook)](https://facebook.github.io/prophet/), [GluonTS (d'Amazon)] (https://ts.gluon.ai/stable/) et bien d'autres bibliothèques. La plupart des bibliothèques exigent un format de données d'entrée précis (mais semblable de l'une à l'autre) et des traitements particuliers. L'étape la plus importante pour utiliser ces paquets consiste donc à préparer les données dans le bon format.

Il existe aussi des bibliothèques libres qui s'appuient sur de nombreux paquets de prévision de séries chronologiques et offrent des interfaces vers une grande variété d'algorithmes de séries chronologiques, comme [sktime](https://www.sktime.net/en/latest/index.html) et [darts](https://unit8co.github.io/darts/). Ces interfaces de prévision de séries chronologiques simplifient grandement la prévision grâce à leur API intuitive et à leur large éventail de modèles, allant des méthodes statistiques classiques aux approches modernes d'apprentissage automatique, y compris les modèles fondés sur l'apprentissage profond.

Dans ce carnet, nous proposons un survol simple de l'analyse de séries chronologiques avec darts. Une démonstration présente également des chaînes de traitement plus complètes du processus de prévision.

Tout d'abord, l'installation de `darts` est obligatoire, car cette bibliothèque n'est pas incluse dans Colab par défaut.

In [ ]:
# Celle-ci installe la version complète de darts.
# Si cela provoque une erreur mentionnant « numpy », cliquez sur « Runtime -> Restart session », puis réexécutez le carnet.
!pip install darts
!pip install statsforecast

# Installer les dépendances manquantes
!pip install pytorch-lightning

# voici une autre option d'installation qui pourrait fonctionner
# !pip install "u8darts[torch]"

### **Étape 1** : Charger les données

Pour utiliser `darts`, il faut un objet particulier appelé `TimeSeries`. Celui-ci est essentiellement très semblable à un `DataFrame` en format de série chronologique : il suffit de passer la série chronologique sous forme de `Series` ou de `DataFrame` pour créer l'objet `TimeSeries` de `darts`. **IMPORTANT** : l'index de la `Series` ou du `DataFrame` doit, avant la conversion, être au format `datetime` (c'est-à-dire en utilisant la fonction `pd.to_datetime(...)`).

In [ ]:
# code Python simple pour les séries chronologiques avec darts

import pandas as pd
import darts

data = pd.read_csv('https://bit.ly/m5simple', index_col='ds')
data.index = pd.to_datetime(data.index)

y_timeseries = darts.TimeSeries.from_series(data['y'])

### **Étape 2** : Séparation entraînement/test

Nous séparons ensuite les données de la série chronologique en un ensemble d'entraînement et un ensemble de test. L'ensemble de test contient les 52 derniers points de données, tandis que l'ensemble d'entraînement contient toutes les données depuis le début jusqu'à la période précédant l'ensemble de test.

In [ ]:
test_n_points = 52

start = len(data)-test_n_points
train, test = y_timeseries.split_before(start)

### **Étape 3** : Entraîner le modèle

Comme avec `sklearn`, il suffit de créer un objet modèle et d'y ajuster les données. Dans le bloc suivant, nous donnons des exemples de différents modèles de séries chronologiques et vous pouvez en choisir un à essayer. La fonction `.fit(...)` sert ensuite à ajuster les données au modèle (entraînement).

La liste des modèles pris en charge par `darts` est disponible ici :
https://unit8co.github.io/darts/generated_api/darts.models.forecasting.html

In [ ]:
from darts.models import ExponentialSmoothing, AutoARIMA, Theta, Prophet, Croston

model = ExponentialSmoothing()
# model = AutoARIMA()
# model = Theta()
# model = Prophet()
# model = Croston()

# Décommentez ci-dessous si vous voulez essayer le modèle d'apprentissage profond NBEATS (il pourrait y avoir un problème de dépendance)
# from darts.models import NBEATSModel
# model = NBEATSModel(input_chunk_length=52, output_chunk_length=52, n_epochs=50)

# Décommentez ci-dessous si vous voulez essayer LightGBMModel
# from darts.models import LightGBMModel
# model = LightGBMModel(lags=52)

# LightGBM est un cadre d'amplification de gradient (gradient boosting) qui utilise des algorithmes d'apprentissage fondés sur des arbres.
# Lorsqu'on utilise LightGBM avec darts, il faut souvent préciser 'lgbm_kwargs' pour transmettre des paramètres
# directement au régresseur LightGBM.
# 'lgbm_kwargs' peut inclure des paramètres comme 'n_estimators', 'learning_rate', 'num_leaves', etc.
# Pour les séries chronologiques, il est crucial de fournir aussi 'lags' afin de préciser quelles valeurs passées de la série
# doivent servir de variables explicatives.

model.fit(train)

### **Étape 4** : Créer les prévisions pour la période de l'ensemble de test

Nous pouvons ensuite générer la prévision des 52 périodes suivantes, ce qui correspond à la période de l'ensemble de test, afin de mesurer la qualité des prévisions sur une base hors échantillon (en utilisant des données de test qui n'ont pas servi à l'entraînement). Nous pouvons aussi convertir la sortie en série (avec `.pd_series()`) ou en dataframe (avec `.pd_dataframe()`).

In [ ]:
forecast = model.predict(len(test))

forecast.to_series()

### **Étape 5** : Mesurer les erreurs de prévision

Nous mesurons les résultats à l'aide de cinq mesures d'erreur différentes, soit l'erreur absolue moyenne en pourcentage (MAPE) avec la fonction `mape()`, l'erreur absolue moyenne pondérée en pourcentage (wMAPE) avec la fonction `smape()`, la racine de l'erreur quadratique moyenne mise à l'échelle (RMSSE) avec la fonction `rmsse()`, la racine de l'erreur quadratique moyenne (RMSE) avec la fonction `rmse()`, et l'erreur moyenne (ME) avec la fonction `merr()`.

In [ ]:
m_mape = darts.metrics.mape(test, forecast)
m_wmape = darts.metrics.wmape(test, forecast)
m_rmsse = darts.metrics.rmsse(test, forecast, insample = train)
m_rmse = darts.metrics.rmse(test, forecast)
m_merr = darts.metrics.merr(test, forecast)


print(f"Le modèle obtient une erreur absolue moyenne en pourcentage : {m_mape:.2f} %")
print(f"Le modèle obtient une erreur absolue moyenne pondérée en pourcentage : {m_wmape:.2f} %")
print(f"Le modèle obtient une racine de l'erreur quadratique moyenne mise à l'échelle : {m_rmsse:.2f}")
print(f"Le modèle obtient une racine de l'erreur quadratique moyenne : {m_rmse:.2f}")
print(f"Le modèle obtient une erreur moyenne : {m_merr:.2f}")

Vous pouvez aussi tracer les résultats avec `seaborn`.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(18,6))

sns.scatterplot(x = data[-104:].index, y = data['y'][-104:].values, label = 'réel')
sns.lineplot(data = forecast.to_series(), label = 'Prévision')

### Note : comprendre l'entrée tabulaire de LightGBM

Lorsqu'on utilise `LightGBMModel` de `darts` avec un paramètre `lags`, le modèle convertit implicitement les données de série chronologique en un format tabulaire où chaque ligne représente un pas de temps et où les colonnes représentent la variable cible et ses valeurs décalées (les variables explicatives). Avec `lags=52`, cela signifie que, pour chaque point de prévision, le modèle considère les 52 valeurs précédentes de la série comme variables explicatives. Comme ce traitement est intégré au paquet, on ne peut pas voir entièrement les données transformées qui sont utilisées lors de l'entraînement.

Pour illustrer cela, le code suivant crée manuellement un `DataFrame` comportant des variables décalées afin d'illustrer ce concept pour notre ensemble `train`. Cela vous montrera le type de données tabulaires avec lequel LightGBM travaille.

In [ ]:
import pandas as pd

# Convertir la TimeSeries darts 'train' en une Series pandas
train_series = train.to_series()

# Créer un DataFrame pour stocker les variables décalées
lagged_df = pd.DataFrame(index=train_series.index)

# Ajouter la variable cible (y) au DataFrame
lagged_df['y'] = train_series

# Définir le nombre de décalages (tel qu'utilisé dans LightGBMModel)
lags_to_generate = 52

# Générer les variables décalées
for i in range(1, lags_to_generate + 1):
    lagged_df['lag_'+str(i)] = train_series.shift(i)

# Afficher les premières lignes du DataFrame de variables décalées généré
# Note : les 'lags_to_generate' premières lignes contiendront des valeurs NaN pour les décalages,
# car il n'y a pas assez de points de données précédents.
display(lagged_df.head(15))
display(lagged_df.tail(15))

